In [4]:
# Quick pos_rate check for different k thresholds on one labeled week
# Run this in Studio / EC2 where AWS creds are available.

import io
import boto3
import numpy as np
import pandas as pd
from urllib.parse import urlparse

S3_ROOT = "s3://smart-park-seattle/parking/"
YEAR = 2022
WEEK = "2022-06-13"   # <-- 아무 week 하나로 바꿔도 됨
GT_PREFIX = f"labeled/year={YEAR}/week={WEEK}/data.csv.gz"

# ---- S3 helpers ----
def parse_s3_uri(s3_uri: str):
    u = urlparse(s3_uri)
    bucket = u.netloc
    key = u.path.lstrip("/")
    if key and not key.endswith("/"):
        key += "/"
    return bucket, key

bucket, root_prefix = parse_s3_uri(S3_ROOT)
key = root_prefix + GT_PREFIX

s3 = boto3.client("s3")
obj = s3.get_object(Bucket=bucket, Key=key)
df = pd.read_csv(io.BytesIO(obj["Body"].read()), compression="gzip", low_memory=False)

# ---- required cols check ----
need = ["paid_occupancy", "space_count"]
for c in need:
    if c not in df.columns:
        raise KeyError(f"Missing {c} in {key}. Have columns: {list(df.columns)[:30]} ...")

paid = pd.to_numeric(df["paid_occupancy"], errors="coerce")
space = pd.to_numeric(df["space_count"], errors="coerce")
m = paid.notna() & space.notna() & (space > 0)
paid = paid[m]
space = space[m]

# clamp + empty_spots (same as preprocess)
paid_clamped = paid.clip(lower=0)
paid_clamped = np.minimum(paid_clamped, space)
empty_spots = (space - paid_clamped).astype(float)

def pos_rate_has_space(k):
    # has_space = empty_spots > k
    return float((empty_spots > k).mean())

def pos_rate_has_space_ratio(alpha):
    # has_space = empty_spots > ceil(alpha * space_count)
    k = np.ceil(alpha * space).astype(float)
    return float((empty_spots > k).mean())

print(f"[Loaded] s3://{bucket}/{key}")
print(f"N={len(empty_spots):,}")
print(f"empty_spots stats:\n{empty_spots.describe(percentiles=[.01,.05,.5,.95,.99]).to_string()}")

# compare k choices
for k in [0, 1, 2, 3, 5]:
    print(f"pos_rate(has_space=empty_spots>{k}) = {pos_rate_has_space(k):.4f}")

for alpha in [0.01, 0.05, 0.10]:
    print(f"pos_rate(has_space=empty_spots>ceil({alpha}*space)) = {pos_rate_has_space_ratio(alpha):.4f}")

[Loaded] s3://smart-park-seattle/parking/labeled/year=2022/week=2022-06-13/data.csv.gz
N=440,549
empty_spots stats:
count    440549.000000
mean          4.281230
std           4.021315
min           0.000000
1%            0.000000
5%            0.000000
50%           3.666667
95%          10.333333
99%          19.000000
max          54.000000
pos_rate(has_space=empty_spots>0) = 0.8962
pos_rate(has_space=empty_spots>1) = 0.7996
pos_rate(has_space=empty_spots>2) = 0.6702
pos_rate(has_space=empty_spots>3) = 0.5291
pos_rate(has_space=empty_spots>5) = 0.2926
pos_rate(has_space=empty_spots>ceil(0.01*space)) = 0.7996
pos_rate(has_space=empty_spots>ceil(0.05*space)) = 0.7993
pos_rate(has_space=empty_spots>ceil(0.1*space)) = 0.7945


#### Empty Spot Mean is 4.57! We want to reduce the pos_rate from 93%.
#### There will always be people who park without paying, disabled individuals who do not need to pay, load/unload (for examples)
#### Also, since we are aggregating data into 15mins bin based on 'last' (not 'mean' nor 'max')

#### In order to reflect these, it's reasonable to select K = 3.
#### on average, when empty_spot > 3, pos_rate remains around 54%